In [1]:
# imports of necessary packages
import torch
import torch.nn as nn
import pytorch_lightning as pl
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
import sys
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from pytorch_lightning import Trainer

#  Hyperparameters
input_size = 784 #28x28
hidden_size1 = 500
hidden_size2 = 200
num_classes = 10
num_epochs = 5
learning_rate = 0.001





In [2]:
# %% Building Feed Forward Neural Network using Lightning

class FeedForwardNetLightning(pl.LightningModule):
    def __init__(self, input_size, hidden_size1, hidden_size2, num_classes):
        super(FeedForwardNetLightning, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1) 
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2) 
        self.fc3 = nn.Linear(hidden_size2, num_classes) 
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out
    
    def training_step(self, batch, batch_idx):
        images, labels = batch
        images = images.reshape(-1, 28*28)
        outputs = self.forward(images)
        loss = F.cross_entropy(outputs, labels)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss
    
    def train_dataloader(self):
        train_dataset = torchvision.datasets.MNIST(root='./data', 
                    train=True, transform=transforms.ToTensor(), download=True)
        train_loader = DataLoader(dataset=train_dataset, 
                    batch_size=100, num_workers=0, shuffle=True)
        return train_loader
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        return optimizer

In [3]:
# Training the model
trainer = Trainer(fast_dev_run=False, max_epochs=num_epochs)
model = FeedForwardNetLightning(input_size, hidden_size1, hidden_size2, num_classes)
trainer.fit(model)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


┏━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ fc1  │ Linear │  392 K │ train │     0 │
│ 1 │ relu │ ReLU   │      0 │ train │     0 │
│ 2 │ fc2  │ Linear │  100 K │ train │     0 │
│ 3 │ fc3  │ Linear │  2.0 K │ train │     0 │
└───┴──────┴────────┴────────┴───────┴───────┘

Trainable params: 494 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 494 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 4                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\USER\anaconda3\envs\pytorch\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


`Trainer.fit` stopped: `max_epochs=5` reached.
